# Fine-tune Qwen3.5-4B for the Decillion dsh agentic system

Run the cells in order (Runtime → Change runtime type → a GPU; A100/L4 recommended, T4 works with shorter context).

1. GPU check
2. Mount Google Drive (outputs go to `MyDrive/decillion-qwen35-4b`)
3. Install libraries
4. Configuration (the only cell you normally edit)
5. Clone the dataset from GitHub and build the training rows
6. Choose the training mix
7. Load Qwen3.5-4B with LoRA
8. Render rows with the model's chat template
9. Trainer with loss on the assistant's turns only
10. Train (checkpoints, including the LoRA adapter, are saved to Drive and resumed after a disconnect)

In [ ]:
# 1. GPU check
!nvidia-smi
import torch
assert torch.cuda.is_available(), "No GPU: Runtime → Change runtime type → GPU"
GPU = torch.cuda.get_device_name(0)
GPU_MEM_GB = torch.cuda.get_device_properties(0).total_memory / 1e9
print(GPU, f"{GPU_MEM_GB:.0f} GB")

In [ ]:
# 2. Google Drive: checkpoints, logs and the final model live here
from google.colab import drive
drive.mount('/content/drive')
import os
OUT_DIR = '/content/drive/MyDrive/decillion-qwen35-4b'
os.makedirs(OUT_DIR, exist_ok=True)
print('outputs →', OUT_DIR)

In [ ]:
# 3. Install (Unsloth brings matching torch / transformers / trl / peft versions)
!pip install -q -U unsloth unsloth_zoo
!pip install -q -U jsonschema
# If loading the model later fails on the Qwen3.5 architecture, run this and restart the runtime:
# !pip install -q -U transformers

In [ ]:
# 4. Configuration (the only cell you normally edit)
MODEL_ID = 'Qwen/Qwen3.5-4B'
REPO = 'https://github.com/keyhmoham/Aseman.git'
BRANCH = 'claude/zen-franklin-krxjsu'
DATA_DIR = '/content/Aseman/datasets/decillion-dsh-agentic'

BIG_GPU = GPU_MEM_GB >= 35
MAX_SEQ_LEN = 16384 if BIG_GPU else 8192
LOAD_IN_4BIT = not BIG_GPU

# LoRA
LORA_R, LORA_ALPHA, LORA_DROPOUT = 32, 32, 0.0

# Training
EPOCHS = 1
LEARNING_RATE = 1e-4
BATCH_SIZE = 2 if BIG_GPU else 1
GRAD_ACCUM = 8 if BIG_GPU else 16
WARMUP_RATIO = 0.03
SAVE_STEPS = 200

# Data mix: multi-turn sessions are ~94% of the data; single-turn families are repeated to balance it
SINGLE_TURN_REPEAT = 4
MAX_MULTI_TURN = 30000   # cap on session.* rows per epoch (None = all ~64k)
TOOLS_MODE = 'relevant'  # 'relevant' (default) or 'full' (all ~130 tool schemas per row; needs long context)
SEED = 3407

print(dict(MODEL_ID=MODEL_ID, MAX_SEQ_LEN=MAX_SEQ_LEN, LOAD_IN_4BIT=LOAD_IN_4BIT, BATCH=BATCH_SIZE, ACCUM=GRAD_ACCUM))

In [ ]:
# 5. Get the dataset from GitHub and build the training rows (about 3 minutes)
import os
if not os.path.exists(DATA_DIR):
    !git clone -q --depth 1 -b {BRANCH} {REPO} /content/Aseman
else:
    !git -C /content/Aseman pull -q
!python3 {DATA_DIR}/scripts/build.py --tools {TOOLS_MODE} --out /content/dist
!ls -la /content/dist
!cat /content/dist/stats.json | head -40

In [ ]:
# 6. Choose the training mix (rows stay raw JSON lines to save memory)
import gzip, json, random, re
from collections import Counter
random.seed(SEED)
CAT = re.compile(r"\"category\": \"([^\"]+)\"")
single, multi = [], []
for line in gzip.open('/content/dist/train.jsonl.gz', 'rt', encoding='utf-8'):
    cat = CAT.search(line[:300]).group(1)
    (multi if cat.startswith('session.') else single).append(line)
random.shuffle(multi)
if MAX_MULTI_TURN:
    multi = multi[:MAX_MULTI_TURN]
train_lines = single * SINGLE_TURN_REPEAT + multi
random.shuffle(train_lines)
val_lines = gzip.open('/content/dist/validation.jsonl.gz', 'rt', encoding='utf-8').read().splitlines()
print(f'single-turn {len(single)} x{SINGLE_TURN_REPEAT}, multi-turn {len(multi)} -> train {len(train_lines)}, validation {len(val_lines)}')

In [ ]:
# 7. Load Qwen3.5-4B and attach LoRA adapters
from unsloth import FastLanguageModel
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_ID,
    max_seq_length=MAX_SEQ_LEN,
    load_in_4bit=LOAD_IN_4BIT,
    dtype=None,
)
model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
    bias='none',
    use_gradient_checkpointing='unsloth',
    random_state=SEED,
)
model.print_trainable_parameters()

In [ ]:
# 8. Render each row with the model's own chat template (tools included) and drop rows that do not fit
from datasets import Dataset
tok = getattr(tokenizer, "tokenizer", tokenizer)

def render(line):
    row = json.loads(line)
    return tokenizer.apply_chat_template(row["messages"], tools=row["tools"], tokenize=False)

def rows(lines):
    for line in lines:
        text = render(line)
        if len(tok(text, add_special_tokens=False).input_ids) <= MAX_SEQ_LEN:
            yield {"text": text}

train_ds = Dataset.from_generator(rows, gen_kwargs={"lines": train_lines})
val_ds = Dataset.from_generator(rows, gen_kwargs={"lines": val_lines})
print('train', len(train_ds), 'of', len(train_lines), '| validation', len(val_ds), 'of', len(val_lines))
print(train_ds[0]["text"][-1500:])

In [ ]:
# 9. Trainer: loss only on the assistant's turns (replies and tool calls), not on system, user or tool results
from trl import SFTTrainer, SFTConfig
from unsloth.chat_templates import train_on_responses_only
from unsloth import is_bfloat16_supported

args = SFTConfig(
    output_dir=f'{OUT_DIR}/checkpoints',
    dataset_text_field='text',
    packing=False,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LEARNING_RATE,
    lr_scheduler_type='cosine',
    warmup_ratio=WARMUP_RATIO,
    weight_decay=0.01,
    optim='adamw_8bit',
    bf16=is_bfloat16_supported(),
    fp16=not is_bfloat16_supported(),
    logging_steps=10,
    eval_strategy='steps',
    eval_steps=SAVE_STEPS,
    save_strategy='steps',
    save_steps=SAVE_STEPS,
    save_total_limit=3,
    report_to='none',
    seed=SEED,
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_ds,
    eval_dataset=val_ds.select(range(min(300, len(val_ds)))),
    args=args,
)
# Qwen chat format: tool results arrive inside user turns (<tool_response>), so masking every
# non-assistant turn leaves exactly the agent's own output in the loss.
trainer = train_on_responses_only(
    trainer,
    instruction_part='<|im_start|>user\n',
    response_part='<|im_start|>assistant\n',
)
print('steps per epoch ≈', len(train_ds) // (BATCH_SIZE * GRAD_ACCUM))

In [ ]:
# 10. Train (resumes from the last checkpoint on Drive if the runtime was disconnected)
import glob
last = sorted(glob.glob(f'{OUT_DIR}/checkpoints/checkpoint-*'), key=lambda p: int(p.rsplit('-', 1)[1]))
stats = trainer.train(resume_from_checkpoint=last[-1] if last else None)
print(stats)